In [1]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns

# Load cleaned data
df = pd.read_csv('../data/processed/telco_churn_cleaned.csv')

# Create SQLite database
conn = sqlite3.connect('../data/telco_churn.db')
df.to_sql('customers', conn, if_exists='replace', index=False)
print("Database created successfully")

# ---- QUERY 1: Churn rate by contract type ----
query1 = """
SELECT 
    Contract,
    COUNT(*) as total_customers,
    SUM(Churn) as churned,
    ROUND(AVG(Churn) * 100, 2) as churn_rate_pct,
    ROUND(AVG(MonthlyCharges), 2) as avg_monthly_charges
FROM customers
GROUP BY Contract
ORDER BY churn_rate_pct DESC
"""
df_q1 = pd.read_sql_query(query1, conn)
print("\nQuery 1 — Churn by Contract:")
print(df_q1)

# ---- QUERY 2: Cohort analysis by tenure group ----
query2 = """
SELECT 
    CASE 
        WHEN tenure BETWEEN 0 AND 12 THEN '0-12 months'
        WHEN tenure BETWEEN 13 AND 24 THEN '13-24 months'
        WHEN tenure BETWEEN 25 AND 36 THEN '25-36 months'
        WHEN tenure BETWEEN 37 AND 48 THEN '37-48 months'
        WHEN tenure BETWEEN 49 AND 60 THEN '49-60 months'
        ELSE '60+ months'
    END as tenure_cohort,
    COUNT(*) as total_customers,
    SUM(Churn) as churned,
    ROUND(AVG(Churn) * 100, 2) as churn_rate_pct,
    ROUND(AVG(MonthlyCharges), 2) as avg_monthly_charges,
    ROUND(AVG(TotalCharges), 2) as avg_total_charges
FROM customers
GROUP BY tenure_cohort
ORDER BY churn_rate_pct DESC
"""
df_q2 = pd.read_sql_query(query2, conn)
print("\nQuery 2 — Cohort Analysis by Tenure:")
print(df_q2)

# ---- QUERY 3: High value churned customers ----
query3 = """
SELECT 
    Contract,
    InternetService,
    PaymentMethod,
    COUNT(*) as churned_customers,
    ROUND(AVG(MonthlyCharges), 2) as avg_monthly_charges,
    ROUND(SUM(MonthlyCharges), 2) as total_monthly_revenue_lost
FROM customers
WHERE Churn = 1
GROUP BY Contract, InternetService, PaymentMethod
ORDER BY total_monthly_revenue_lost DESC
LIMIT 10
"""
df_q3 = pd.read_sql_query(query3, conn)
print("\nQuery 3 — Top Revenue Lost Segments:")
print(df_q3)

# ---- QUERY 4: Window function — running churn rate ----
query4 = """
WITH monthly_stats AS (
    SELECT 
        tenure,
        COUNT(*) as customers,
        SUM(Churn) as churned,
        ROUND(AVG(Churn) * 100, 2) as churn_rate
    FROM customers
    GROUP BY tenure
)
SELECT 
    tenure,
    customers,
    churned,
    churn_rate,
    ROUND(AVG(churn_rate) OVER (
        ORDER BY tenure 
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ), 2) as rolling_avg_churn_rate
FROM monthly_stats
ORDER BY tenure
"""
df_q4 = pd.read_sql_query(query4, conn)
print("\nQuery 4 — Rolling Average Churn Rate by Tenure:")
print(df_q4.head(20))

conn.close()

Database created successfully

Query 1 — Churn by Contract:
         Contract  total_customers  churned  churn_rate_pct  \
0  Month-to-month             3875     1655           42.71   
1        One year             1472      166           11.28   
2        Two year             1685       48            2.85   

   avg_monthly_charges  
0                66.40  
1                65.08  
2                60.87  

Query 2 — Cohort Analysis by Tenure:
  tenure_cohort  total_customers  churned  churn_rate_pct  \
0   0-12 months             2175     1037           47.68   
1  13-24 months             1024      294           28.71   
2  25-36 months              832      180           21.63   
3  37-48 months              762      145           19.03   
4  49-60 months              832      120           14.42   
5    60+ months             1407       93            6.61   

   avg_monthly_charges  avg_total_charges  
0                56.17             276.62  
1                61.36           